In [1]:
import torch
import math

class Dropout:
    
    def __init__(self,p=0.1,training=True):
        self.p = p
        self.training = training
        self.mask = None
        

    def forward(self,x):
        if self.training:
            self.mask = ((torch.rand_like(x) > self.p).float())/(1.0-self.p)
            return x * self.mask
        else:
            self.mask = None
            return x   

    def backward(self,grad_out):
        if self.training:
            return grad_out * self.mask
        else:
            return grad_out



class LayerNorm:

    def __init__(self,num_dims,eps):
        self.eps = eps
        self.gamma = torch.ones(num_dims)
        self.beta = torch.zeros(num_dims)


    def forward(self,x):
        self.avg = torch.mean(x,dim=-1,keepdim=True)
        self.var = torch.mean( (x - self.avg)**2,dim=-1,keepdim=True )

        self.x_norm = (x - self.avg)/torch.sqrt(self.var+self.eps)
        out = self.x_norm * self.gamma + self.beta
        return out

    
    def backward(self, grad_out):
        self.grad_beta = torch.sum(grad_out, dim=(0, 1))
        self.grad_gamma = torch.sum(grad_out * self.x_norm, dim=(0, 1))
    
        grad_x = grad_out * self.gamma

        return (1.0 / torch.sqrt(self.var + self.eps)) * (
            grad_x
            - torch.mean(grad_x, dim=-1, keepdim=True)
            - self.x_norm * torch.mean(grad_x * self.x_norm, dim=-1, keepdim=True)
        )
        




        
        

# Dropout & Layer Normalization

We use a special method called **Dropout**. We are often encouraged to use Dropout to randomly deactivate a percentage of the neurons, because some neurons can be lazy enough to work and try to copy other neurons' work.

For example:
- $y_1 \to$ feature of neuron 1
- $y_2 \to$ feature of neuron 2
- $y_3 \to y_1 + y_2$

Here $y_3$ isn't actually creating a new feature—it is just using other neurons' work so we can drop (fuse) this neuron. But in the model, we can't know beforehand which neuron is being redundant so we drop them randomly to force them to learn features by themselves.

## Where do we actually use Dropout?

### 1. Attention (After Softmax)

Okay wait, why do we need to use it after Softmax? Because there is a clear reason for that:

If you use it before Softmax, those zeroed values become $0$ and $e^0 = 1$ at Softmax—they will revive using reincarnation jutsu 😂️... so we don't want them coming back alive!

After Softmax, we have already done the causal masking for the next tokens so probabilities sum up to a perfect 1. This is the perfect place to use Dropout.

Because we are about to multiply with $V$ (which holds the real content), before the model learns to fetch $V$, we need to make sure the model uses all feature dimensions properly. So we use Dropout at that place.

It prevents the model from obsessing over just one specific token in the context. By randomly zeroing out attention connections, it forces the query token to learn to attend to multiple different context tokens.

### 2. Residual Connection (Before Residual Addition)

And the next position where we actually apply Dropout is just before adding the residual output back to the original tokens.

You might encounter these in the future as we step down a few blocks so you can skip this part and come back later too. We will soon understand why we actually mix them up (spoilers: because we still need the original $x$ alongside predictions to help the model predict words better).

### 3. MLP Stage

And at the MLP stage, we use Dropout before adding back to $x$.

### 4. Model Entrance (After Embeddings)

At the very entrance of the model, right after combining token embeddings and positional embeddings:

$$x = \text{tok\_emb} + \text{pos\_emb}$$

$$x = \text{dropout}(x)$$

## Layer Normalization (LayerNorm)

Now coming to the real LayerNorm work. The reason for LayerNorm is to ensure that activations don't either blow up or vanish.

Gemini generated this example so well, check this out:

### The Microphone and Amplifier Problem

Imagine a chain of 12 speakers and amplifiers in a room:
- The first amplifier boosts the audio signal by just $1.2\times$.
- The next amplifier boosts it by $1.2\times$.
- By the 12th amplifier, the sound volume is $1.2^{12} \approx 8.9\text{x}$ louder.
- If one amp slips and boosts by $2\times$, $2^{12} = 4096\text{x}$. The sound turns into deafening screeches and distortion.

In a deep neural network, every layer multiplies matrices and adds residual vectors. As activations travel deeper through the network:
- Without control, numbers either blow up toward infinity (**exploding activations**) or shrink toward zero (**vanishing activations**).

When values blow up, mathematical operations like $\text{Softmax}$ choke: the largest number gets a probability of $1.0$, all others become $0.0$ and gradients drop dead to zero.

LayerNorm acts as an automatic sound engineer. Right before the signal enters an Attention or MLP block, LayerNorm grabs that specific token vector, centers its volume to 0 and normalizes its loudness to 1.

## Where is it placed? (Pre-LN vs. Post-LN)

In early Transformers (original "Attention is All You Need"), normalization was applied after the residual add (Post-LN):

$$x = \text{LayerNorm}(x + \text{SubLayer}(x))$$

This caused training instability because backpropagating gradients had to fight through normalization at every single residual step.

Modern architectures like GPT-2, GPT-3 and LLaMA use Pre-LN:

$$x = x + \text{SubLayer}(\text{LayerNorm}(x))$$

In Pre-LN:
- **Clean Highway**: The residual stream $x$ remains an untouched, clean highway where gradients can flow from the end of the model all the way to token embeddings without restriction.
- **Safe Input**: Each sub-layer (Attention and MLP) receives an input that is cleanly scaled and normalized so it never receives exploding numbers.

## LayerNorm Concept & Mathematics

In LayerNorm, single tokens have the entire universe. It's like sending a single token, finding the mean and variance on that token, implementing those calculations and sending out that single token. So this happens for all the tokens inside.

### Backward Pass for LayerNorm

When backpropagating through LayerNorm, upstream gradient $\text{grad\_out}$ (let's call it $d_{\text{out}}$) arrives first.

The forward equation was:

$$\text{out} = \hat{x} \odot \gamma + \beta$$

where $\hat{x}$ is the normalized input, $\gamma$ is the learnable scale (starts at $1$) and $\beta$ is the learnable shift (starts at $0$).

### 1. Gradients for Parameters ($\gamma$ and $\beta$)

Because $\text{out}$ is linear with respect to $\beta$ and $\gamma$, their gradients follow the standard bias and weight rules:

* **For $\beta$:** $\frac{\partial \text{out}}{\partial \beta} = 1$

$$\text{grad\_beta} = \sum_{B, T} d_{\text{out}} \implies \text{torch.sum}(\text{grad\_out}, \text{dim}=(0, 1))$$

* **For $\gamma$:** $\frac{\partial \text{out}}{\partial \gamma} = \hat{x}$

$$\text{grad\_gamma} = \sum_{B, T} (d_{\text{out}} \odot \hat{x}) \implies \text{torch.sum}(\text{grad\_out} \cdot \text{self.x\_norm}, \text{dim}=(0, 1))$$

### 2. Gradient Entering the Normalized Input ($d\hat{x}$)

Before touching $x$, route through $\gamma$:

$$d\hat{x} = \frac{\partial \mathcal{L}}{\partial \hat{x}} = d_{\text{out}} \odot \gamma$$

In code:

```python
grad_x_norm = grad_out * self.gamma

```

### 3. Deriving $\frac{\partial \mathcal{L}}{\partial x}$ Using Simple Product Rule

Recall:

$$\hat{x} = \frac{x - \mu}{\sigma_{\epsilon}} = (x - \mu) \cdot \frac{1}{\sigma_{\epsilon}} = u \cdot v$$

where:

* $u = x - \mu$ (centered numerator)
* $v = \frac{1}{\sigma_{\epsilon}} = (\sigma^2 + \epsilon)^{-1/2}$ (scale factor)

When $x_i$ shifts, it influences the output through three distinct paths:

1. **Directly via $x_i$ in the numerator**
2. **Indirectly via the batch mean $\mu$**
3. **Indirectly via the variance $\sigma^2$**

#### Term 1: The Direct Numerator Effect ($u' \cdot v$)

Differentiating $u_i = x_i - \mu$ with respect to $x_i$:

$$\frac{\partial (x_i - \mu)}{\partial x_i} = 1 - \frac{1}{D}$$

Across all inputs, this contributes two simple pieces to the gradient:

* The direct gradient: $d\hat{x}$
* Minus the average gradient across the row: $-\text{mean}(d\hat{x})$

#### Term 2: The Variance Scaling Effect ($u \cdot v'$)

Differentiating scale factor $v = (\sigma^2 + \epsilon)^{-1/2}$ with respect to $x_i$ using the chain rule:

$$\frac{\partial v}{\partial x_i} = \frac{\partial v}{\partial \sigma^2} \cdot \frac{\partial \sigma^2}{\partial x_i} = \left(-\frac{1}{2 \sigma_{\epsilon}^3}\right) \cdot \left(\frac{2}{D}(x_i - \mu)\right) = -\frac{1}{D \sigma_{\epsilon}^2} \cdot \hat{x}_i$$

Multiplying by upstream gradient and accumulating across the feature dimension yields the projection term:

$$-\hat{x} \odot \text{mean}(d\hat{x} \odot \hat{x})$$

This term penalizes updates that simply scale the variance without adding useful directional signal.

### 4. The Final Combined Formula

Pulling the common factor $v = \frac{1}{\sigma_{\epsilon}} = \frac{1}{\sqrt{\sigma^2 + \epsilon}}$ outside leaves three intuitive parts:

$$\frac{\partial \mathcal{L}}{\partial x} = \frac{1}{\sqrt{\sigma^2 + \epsilon}} \Big( \underbrace{d\hat{x}}_{\text{raw gradient}} - \underbrace{\text{mean}(d\hat{x})}_{\text{remove mean shift}} - \underbrace{\hat{x} \odot \text{mean}(d\hat{x} \odot \hat{x})}_{\text{remove variance scaling}} \Big)$$

* **Raw gradient:** Pushes $x$ in the direction that lowers loss.
* **$-\text{mean}(d\hat{x})$:** Keeps the output zero-centered (LayerNorm invariant to shifts).
* **$-\hat{x} \odot \text{mean}(d\hat{x} \odot \hat{x})$:** Keeps output variance constant (LayerNorm invariant to scaling).

### The Code Implementation

```python
# 1. Incoming gradient through gamma
grad_x_norm = grad_out * self.gamma

# 2. Vectorized backward pass
inv_std = 1.0 / torch.sqrt(self.var + self.eps)
dx = inv_std * (
 grad_x_norm
 - torch.mean(grad_x_norm, dim=-1, keepdim=True)
 - self.x_norm * torch.mean(grad_x_norm * self.x_norm, dim=-1, keepdim=True)
)

```